## book_p30 — cross-entropy loss curves for the five book parameter probes

Loads the five P(correct) curves from `p37_param_k/summary.json` (written by `P25_37d_book`) and adds the cross-entropy
loss, `−ln P`. Exports **still** figures only, as transparent-background SVGs. Every figure shares **one** crop box, so
the axes sit in identical positions across the prob-only, loss-only and combined versions.

```
p30_loss/param_k/prob.svg        P(correct) curve, left y-ticks, grid
p30_loss/param_k/loss.svg        loss curve only, left y-ticks on the loss scale, x-axis kept, no grid
p30_loss/param_k/combined.svg    P(correct) at reduced alpha (left axis, grid) + loss on a right axis
p30_loss/param_k/summary.json    y-limits, color, w0, curve data (theta, P, loss)
```

In [ ]:
BLUE="#2ca3dd"
YELLOW="#ffd35a"
RED='#ec2027'
CHILL_BROWN="#948979"

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.figure import Figure                      # OO figures: never registered with the notebook backend
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.transforms import Bbox

plt.rcParams['mathtext.fontset'] = 'cm'
plt.rcParams['font.family'] = 'serif'
plt.rcParams['svg.fonttype'] = 'none'   # editable text in the SVGs ('path' to outline glyphs instead)

GRAPHICS  = Path("/home/stephen/Stephencwelch Dropbox/welch_labs/ai_book_vol_2/3_resnets/graphics")
PARAM_DIR = lambda k: GRAPHICS / f'p37_param_{k}'        # written by P25_37d_book
OUT       = GRAPHICS / 'p30_loss'
SPAN      = 2.5
N_PARAMS  = 5

### Load curves, compute loss

In [ ]:
LOSS_PAD_FRAC = 0.05          # y-padding as a fraction of the loss range (the prob axis uses the same 5%)
P_FLOOR       = 1e-12         # clip before the log so P=0 can't give inf

params = {}
for k in range(1, N_PARAMS + 1):
    meta = json.load(open(PARAM_DIR(k) / 'summary.json'))
    vals = np.array(meta['curve']['theta']); res = np.array(meta['curve']['p_correct'])
    loss = -np.log(np.clip(res, P_FLOOR, None))
    lo, hi = loss.min(), loss.max(); pad = LOSS_PAD_FRAC * (hi - lo)
    plo, phi = res.min(), res.max(); ppad = LOSS_PAD_FRAC * (phi - plo)
    params[k] = dict(k=k, vals=vals, res=res, loss=loss, loss_ylim=(lo - pad, hi + pad), prob_ylim=(plo - ppad, phi + ppad),
                     w0=meta['w0'], color=meta['color'], theta_label=meta['theta_label'], coord=meta['coord'],
                     layer_num=meta['layer_num'], probes=meta['probes'])

print(f"{'k':>2} {'coord':<22} {'color':<8} {'P range':<18} {'loss range':<18} loss ylim")
for p in params.values():
    print(f"{p['k']:>2} {str(tuple(p['coord'])):<22} {p['color']:<8} {p['res'].min():.4f} .. {p['res'].max():.4f}    "
          f"{p['loss'].min():.3f} .. {p['loss'].max():.3f}    {p['loss_ylim'][0]:.3f} .. {p['loss_ylim'][1]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for p in params.values():
    axes[0].plot(p['vals'], p['res'],  c=p['color'], lw=2, label=f"{p['k']}: {p['theta_label']}")
    axes[1].plot(p['vals'], p['loss'], c=p['color'], lw=2)
axes[0].set_title('P(correct)'); axes[1].set_title(r'cross-entropy  $-\ln P$')
for ax in axes: ax.set_xlim(-SPAN, SPAN); ax.grid(alpha=0.3)
axes[0].legend(fontsize=8);

### Renderers (all transparent background)

In [ ]:
# geometry identical to the p37 curve figures
FIGSIZE     = (6, 6)
PROB_YLIM   = (-0.05, 1.05)   # fixed alternative to the adaptive p['prob_ylim']
CURVE_LW    = 3
TICK_FS     = 12
PROB_ALPHA_COMBINED = 0.25

def style_ax(ax, color=CHILL_BROWN):
    ax.spines[:].set_visible(False)
    ax.tick_params(colors=color, which='both', labelsize=TICK_FS)

def new_fig():
    fig = Figure(figsize=FIGSIZE)
    ax = fig.add_subplot(111)
    ax.set_facecolor('none')
    ax.set_xlim(-SPAN, SPAN)
    style_ax(ax)
    return fig, ax

def fig_prob(p):
    """P(correct) curve, grid, left ticks -- same look as the p37 curve figures."""
    fig, ax = new_fig()
    ax.plot(p['vals'], p['res'], lw=CURVE_LW, c=p['color'])
    ax.grid(True, color=CHILL_BROWN, alpha=0.3, linewidth=0.8)
    ax.set_ylim(*p['prob_ylim'])   # or ax.set_ylim(*PROB_YLIM) for a fixed 0..1 axis
    return fig

def fig_loss(p):
    """Loss curve only, left y-ticks on the loss scale, x-axis kept, no grid."""
    fig, ax = new_fig()
    ax.plot(p['vals'], p['loss'], lw=CURVE_LW, c=p['color'])
    ax.set_ylim(*p['loss_ylim'])
    return fig

def fig_combined(p):
    """P(correct) at reduced alpha on the left axis with grid, loss on a right axis."""
    fig, ax = new_fig()
    ax.plot(p['vals'], p['res'], lw=CURVE_LW, c=p['color'], alpha=PROB_ALPHA_COMBINED)
    ax.grid(True, color=CHILL_BROWN, alpha=0.3, linewidth=0.8)
    ax.set_ylim(*p['prob_ylim'])
    ax2 = ax.twinx()
    ax2.plot(p['vals'], p['loss'], lw=CURVE_LW, c=p['color'])
    ax2.set_ylim(*p['loss_ylim'])
    style_ax(ax2)
    ax2.patch.set_alpha(0)
    return fig

VARIANTS = {'prob': fig_prob, 'loss': fig_loss, 'combined': fig_combined}

def save_svg(fig, path, bbox):
    fig.savefig(path, format='svg', bbox_inches=bbox, transparent=True)
    fig.clear()

### One crop box for everything

In [ ]:
def tight(fig):
    FigureCanvasAgg(fig)
    return fig.get_tightbbox(fig.canvas.get_renderer())

boxes = []
for p in params.values():
    for make in VARIANTS.values():
        f = make(p); boxes.append(tight(f)); f.clear()
BBOX = Bbox.union(boxes).padded(0.1)              # savefig(bbox_inches='tight') pads by 0.1 in too
print('crop (inches):', np.round(BBOX.bounds, 3))

### Preview: all three variants, all five parameters

In [ ]:
fig, axes = plt.subplots(N_PARAMS, 3, figsize=(9, 2.6 * N_PARAMS))
for r, p in enumerate(params.values()):
    axes[r, 0].plot(p['vals'], p['res'], c=p['color'], lw=2); axes[r, 0].set_ylim(*p['prob_ylim']); axes[r, 0].grid(alpha=0.3)
    axes[r, 1].plot(p['vals'], p['loss'], c=p['color'], lw=2); axes[r, 1].set_ylim(*p['loss_ylim'])
    axes[r, 2].plot(p['vals'], p['res'], c=p['color'], lw=2, alpha=PROB_ALPHA_COMBINED); axes[r, 2].set_ylim(*p['prob_ylim']); axes[r, 2].grid(alpha=0.3)
    ax2 = axes[r, 2].twinx(); ax2.plot(p['vals'], p['loss'], c=p['color'], lw=2); ax2.set_ylim(*p['loss_ylim'])
    axes[r, 0].set_ylabel(f"{p['k']}: {p['theta_label']}")
    for ax in axes[r]: ax.set_xlim(-SPAN, SPAN)
for ax, t in zip(axes[0], ['P(correct)', r'$-\ln P$', 'combined']): ax.set_title(t)
plt.tight_layout()

## Export

In [ ]:
def write_meta(p, d):
    json.dump({'param': p['k'], 'coord': list(p['coord']), 'layer_num': p['layer_num'], 'theta_label': p['theta_label'],
               'w0': p['w0'], 'color': p['color'], 'span': SPAN, 'loss_pad_frac': LOSS_PAD_FRAC, 'p_floor': P_FLOOR,
               'prob_ylim': list(map(float, p['prob_ylim'])), 'loss_ylim': list(map(float, p['loss_ylim'])),
               'crop_inches': list(map(float, BBOX.bounds)), 'source': str(PARAM_DIR(p['k']) / 'summary.json'),
               'curve': {'theta': p['vals'].tolist(), 'p_correct': p['res'].tolist(), 'loss': p['loss'].tolist()}},
              open(d / 'summary.json', 'w'), indent=2)

for p in params.values():
    d = OUT / f"param_{p['k']}"; d.mkdir(parents=True, exist_ok=True)
    for name, make in VARIANTS.items():
        save_svg(make(p), d / f'{name}.svg', BBOX)
    write_meta(p, d)
    print(d, '->', ', '.join(f'{n}.svg' for n in VARIANTS))